# Install rasterio

In [ ]:
!pip install rasterio

# Import some packages

In [ ]:
import numpy as np
from matplotlib import pyplot as pl
import os
import rasterio
from rasterio.plot import show

# Set the path in which you saved the test image

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
PATH = 'drive/MyDrive/sentinel'
os.listdir(PATH)

# Load the images

In [ ]:
zone_name = 'antioche'
sentinel_images = {}
year_str_len = 4
file_extension_str_len = len('.tif')

for filename in os.listdir(PATH):
    if filename.startswith(zone_name):
        fp = os.path.join(PATH, filename)
        img = rasterio.open(fp).read()
        img *= 255 # Rescale between 0 and 255
        img = np.moveaxis(img, 0, -1) # From channel first to channel last format
        img = np.clip(np.float32(img) * 2, 0, 255).astype(np.uint8) # Rescale brightness and ensure values are between 0 and 255
        key = filename[-(year_str_len + file_extension_str_len):-file_extension_str_len]

        sentinel_images[key] = img
        print(filename, key, sentinel_images[key].shape)

In [ ]:
for year, image in sentinel_images.items():
    print(year, np.min(image), np.max(image))

# Visualize the images

In [ ]:
pl.figure(figsize=(12,16))
i = 1
for year, image in sentinel_images.items():
    pl.subplot(2,2,i)
    pl.imshow(image, vmin=0, vmax=1)
    pl.axis('off')
    i += 1

# Import some packages

In [ ]:
import tensorflow as tf
import cv2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, fbeta_score, accuracy_score

# Create a dataset of small windows from the test image

In [ ]:
# Define constants
INPUT_SHAPE = (64, 64, 3)
WIDTH = INPUT_SHAPE[1]
HEIGHT = INPUT_SHAPE[0]
STRIDE_FACTOR = 8  # step is WINDOW_WIDTH / STRIDE_FACTOR, WINDOW_HEIGHT / STRIDE_FACTOR

# Initialize datasets dictionary
datasets = {}

for year, image in sentinel_images.items():
    # Create sliding window views of the image
    window_shape = (HEIGHT, WIDTH, image.shape[2])
    window_views = np.lib.stride_tricks.sliding_window_view(image, window_shape)
    window_views = window_views[::HEIGHT // STRIDE_FACTOR, ::WIDTH // STRIDE_FACTOR, :]

    # Preprocess the window views
    window_views = preprocess_input(np.float32(window_views))

    datasets[year] = np.squeeze(window_views)
    print(year, datasets[year].shape)

# Predict earthquake damage

In [ ]:
LABELS = ['Intact','Destroyed']

In [ ]:
model = tf.keras.models.load_model('vgg16_earthquakeoct2026.keras')

In [ ]:
outputs = {}

for year, dataset in datasets.items():
    print(year)

    # Reshape the dataset to match the expected shape
    num_window_rows, num_window_cols, num_rows, num_cols, num_channels = dataset.shape
    reshaped_dataset = dataset.reshape((num_window_rows * num_window_cols, num_rows, num_cols, num_channels))

    # Predict using the model
    predictions = model.predict(reshaped_dataset)

    # Reshape the predictions back to the original shape
    reshaped_predictions = predictions.reshape((num_window_rows, num_window_cols))

    outputs[year] = reshaped_predictions

# Visualize the earthquake damage prediction

In [ ]:
import matplotlib.pyplot as plt

# Create a figure for plotting
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 4))

i = 0
for year, output in outputs.items():
    prediction = cv2.resize(output, (num_window_cols, num_window_rows))

    ax = axes.flat[i]
    im = ax.imshow(prediction, vmin=0, vmax=1, cmap="Reds")
    ax.set_axis_off()
    ax.set_title(str(year))
    i += 1

# Add a colorbar
cbar = fig.colorbar(im, ax=axes.ravel().tolist(), ticks=[0, 1])
cbar.ax.set_yticklabels(LABELS)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import ImageGrid

# Create the second set of plots using ImageGrid
fig = plt.figure(figsize=(12, 16))
grid = ImageGrid(fig, 111,  # 111 is the grid location (just one grid in this case)
                 nrows_ncols=(2, 2),
                 axes_pad=0.3,  # Adjust this value to control the padding between subplots
                 cbar_mode="single",  # Create a single colorbar for the subplot
                 cbar_location="right",
                 cbar_size="5%",
                 cbar_pad=0.1)

i = 0
for (year, image), output in zip(sentinel_images.items(), outputs.values()):
    heatmap = cv2.resize(output, (image.shape[1], image.shape[0]))

    ax0 = grid[i]
    ax0.set_title(str(year))
    ax0.imshow(image, vmin=0, vmax=255)
    ax0.axis('off')

    ax1 = grid[i+1]
    ax1.set_title(str(year))
    ax1.imshow(image, vmin=0, vmax=255)
    im = ax1.imshow(heatmap, cmap="Reds", alpha=0.4, vmin=0, vmax=1)  # Set transparency here
    ax1.axis('off')

    cbar = ax1.cax.colorbar(im, ticks=[0,1])
    cbar.ax.set_yticklabels(['Intact','Destroyed'])

    i += 2

# Show the combined figure
plt.show()

## TODO 
Choose a new region and analyze the results of the model on that region